**To run without Jupyter**: Execute `python notebooks/run_in_depth_analysis.py` from the project root. This produces the same output and updates `results.md`.

# In-Depth Analysis: Wildfire Property Intelligence

Comprehensive analysis of all datasets. Outputs full context to `results.md` for LLM consumption.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

# Project root (notebook is in notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT_MD = PROJECT_ROOT / 'results.md'
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'in_depth_analysis'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

analysis_output = []
figures_generated = []

def log(msg):
    """Append to analysis output and print."""
    analysis_output.append(msg)
    print(msg)

In [2]:
def analyze_df(name, df, describe=True, value_counts_cols=None):
    """Analyze a DataFrame and return stats dict."""
    stats = {
        'name': name,
        'shape': df.shape,
        'rows': len(df),
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict(),
        'null_counts': df.isnull().sum().to_dict(),
    }
    if describe and df.select_dtypes(include=[np.number]).shape[1] > 0:
        stats['numeric_summary'] = df.describe().to_dict()
    if value_counts_cols:
        for col in value_counts_cols:
            if col in df.columns:
                vc = df[col].value_counts().head(15)
                stats[f'value_counts_{col}'] = vc.to_dict()
    return stats

def section(title, level=2):
    log('\n' + '#' * level + ' ' + title + '\n')

def table_to_md(df, max_rows=20):
    """Convert DataFrame to markdown table."""
    sub = df.head(max_rows)
    return sub.to_markdown(index=False) if hasattr(sub, 'to_markdown') else sub.to_string()

## 1. Main Dataset

In [3]:
section('1. Main Dataset: Capstone2025_nsi_lvl9_with_landcover_and_color', 2)

df_main = None
main_paths = [
    PROJECT_ROOT / 'dataset' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv',
    PROJECT_ROOT / 'dataset' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz',
    PROJECT_ROOT / 'website' / 'backend' / 'data' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv',
]
main_path = next((p for p in main_paths if p.exists()), None)

if main_path:
    kwargs = {'compression': 'gzip'} if str(main_path).endswith('.gz') else {}
    df_main = pd.read_csv(main_path, low_memory=False, nrows=500000, **kwargs)
    log(f"**File**: `{main_path.relative_to(PROJECT_ROOT)}`")
    log(f"**Shape**: {df_main.shape[0]:,} rows (sampled), {df_main.shape[1]} columns")
    log(f"**Columns**: {list(df_main.columns)}")
    log(f"**Unique H3 cells**: {df_main['h3'].nunique():,}")
    log(f"**Unique counties (FIPS)**: {df_main['fips'].nunique()}")
    log(f"**Duplicates at H3×landcover**: {df_main.duplicated(subset=['h3','lc_type']).sum():,}")
    log(f"**Landcover distribution (top 5)**:")
    for row in df_main['lc_type'].value_counts().head(5).items():
        pct = 100 * row[1] / len(df_main)
        log(f"  - {row[0]}: {row[1]:,} ({pct:.1f}%)")
    log(f"**Color distribution (top 5)**:")
    for row in df_main['clr'].value_counts().head(5).items():
        pct = 100 * row[1] / len(df_main)
        log(f"  - {row[0]}: {row[1]:,} ({pct:.1f}%)")
    log(f"**Error tokens**: foo={len(df_main[df_main['clr']=='foo']):,}, bar={len(df_main[df_main['clr']=='bar']):,}")
else:
    log("Main dataset not found.")


## 1. Main Dataset: Capstone2025_nsi_lvl9_with_landcover_and_color

**File**: `dataset\Capstone2025_nsi_lvl9_with_landcover_and_color.csv`
**Shape**: 500,000 rows (sampled), 8 columns
**Columns**: ['h3', 'fips', 'st_damcat', 'bldgtype', 'lc_type', 'loc', 'clr', 'clr_cc']
**Unique H3 cells**: 51,511
**Unique counties (FIPS)**: 11
**Duplicates at H3×landcover**: 448,489
**Landcover distribution (top 5)**:
  - urban: 284,418 (56.9%)
  - urban+forest: 64,252 (12.9%)
  - forest: 49,131 (9.8%)
  - urban+shrub: 28,855 (5.8%)
  - urban+grass: 25,301 (5.1%)
**Color distribution (top 5)**:
  - cocoa: 63,614 (12.7%)
  - orange: 35,280 (7.1%)
  - olive: 35,009 (7.0%)
  - red: 30,144 (6.0%)
  - terracotta: 29,877 (6.0%)
**Error tokens**: foo=19,857, bar=1,149


## 2. Results/Tables: Exposure Density & Sparsity

In [5]:
section('2. Results: Exposure Density & Sparsity', 2)

exp_dir = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity'
if exp_dir.exists():
    for f in sorted(exp_dir.glob('*.csv')):
        df = pd.read_csv(f)
        log(f"\n### Table: `{f.name}`")
        log(f"- Shape: {df.shape}")
        log(f"- Columns: {list(df.columns)}")
        if 'total_exposure' in df.columns:
            log(f"- Exposure: min={df['total_exposure'].min()}, max={df['total_exposure'].max()}, median={df['total_exposure'].median():.1f}")
        if 'exposure_bin' in df.columns and 'n_h3_cells' in df.columns:
            log(f"- Sparsity regimes:")
            for _, r in df.iterrows():
                log(f"  - {r['exposure_bin']}: {r['n_h3_cells']:,} cells ({r['pct_h3_cells']}%), {r['total_structures']:,} structures")
        if 'landcover' in df.columns:
            log(f"- Landcover stats (sample):")
            for _, r in df.head(5).iterrows():
                log(f"  - {r['landcover']}: median={r.get('median_exposure', 'N/A')}, pct_low={r.get('pct_low_exposure', 'N/A')}")
        log('')
else:
    log("Exposure directory not found.")


## 2. Results: Exposure Density & Sparsity


### Table: `eda_exposure_by_county.csv`
- Shape: (58, 4)
- Columns: ['county_fips', 'n_h3_cells', 'total_exposure', 'median_exposure']
- Exposure: min=300, max=445121, median=16797.5


### Table: `eda_exposure_by_landcover.csv`
- Shape: (13, 8)
- Columns: ['landcover', 'median_exposure', 'mean_exposure', 'std_exposure', 'min_exposure', 'max_exposure', 'iqr', 'pct_low_exposure']
- Landcover stats (sample):
  - urban: median=16.0, pct_low=12.129360143011212
  - urban+crop: median=16.0, pct_low=33.48024948024948
  - urban+grass: median=16.0, pct_low=19.65972520395019
  - urban+barren: median=15.0, pct_low=29.43698199485567
  - urban+forest: median=13.0, pct_low=32.00689503410841


### Table: `eda_exposure_diversity.csv`
- Shape: (221108, 5)
- Columns: ['h3', 'n_colors', 'n_occupancy', 'n_bldgtype', 'total_exposure']
- Exposure: min=1, max=71, median=12.0


### Table: `eda_exposure_per_h3.csv`
- Shape: (221108, 3)
- Columns: ['h3', 'total_expos

## 3. Results/Tables: Color Analysis

In [6]:
section('3. Results: Color Analysis', 2)

color_dir = PROJECT_ROOT / 'results' / 'tables' / '03_color'
if color_dir.exists():
    for f in sorted(color_dir.glob('*.csv')):
        df = pd.read_csv(f)
        log(f"\n### Table: `{f.name}`")
        log(f"- Shape: {df.shape}")
        log(f"- Columns: {list(df.columns)}")
        log('')
else:
    log("Color directory not found.")


## 3. Results: Color Analysis


### Table: `color_similarity_matrix.csv`
- Shape: (38, 39)
- Columns: ['clr', 'alabaster', 'amber', 'aqua', 'aquamarine', 'auburn', 'azure', 'bar', 'beige', 'blue', 'brown', 'cocoa', 'coffee', 'crimson', 'emerald', 'foo', 'gold', 'gray', 'green', 'grey', 'indigo', 'ivory', 'lavender', 'lemon', 'lilac', 'maroon', 'navy', 'olive', 'orange', 'plum', 'purple', 'red', 'sage', 'scarlet', 'sienna', 'tan', 'terracotta', 'verde', 'yellow']


### Table: `landcover_color_combinations.csv`
- Shape: (419, 6)
- Columns: ['lc_type', 'clr', 'count', 'total', 'proportion', 'stable']



## 4. Results/Tables: Bayesian Shrinkage

In [7]:
section('4. Results: Bayesian Shrinkage', 2)

bay_dir = PROJECT_ROOT / 'results' / 'tables' / 'bayesian_shrinkage'
if bay_dir.exists():
    for f in sorted(bay_dir.glob('*.csv')):
        df = pd.read_csv(f)
        log(f"\n### Table: `{f.name}`")
        log(f"- Shape: {df.shape}")
        log(f"- Columns: {list(df.columns)}")
        if 'baseline_prop' in df.columns:
            log(f"- Landcover types: {df['lc_type'].nunique()}")
            log(f"- Categories per landcover: ~{len(df) // df['lc_type'].nunique()}")
        if 'stabilized_prop' in df.columns:
            log(f"- Exposure range: {df['exposure'].min()} - {df['exposure'].max()}")
            log(f"- Shrinkage weight range: {df['shrinkage_weight'].min():.4f} - {df['shrinkage_weight'].max():.4f}")
        log('')
else:
    log("Bayesian directory not found.")


## 4. Results: Bayesian Shrinkage


### Table: `bayesian_shrinkage_aggregated_counts.csv`
- Shape: (4493, 5)
- Columns: ['fips', 'lc_type', 'clr', 'count', 'exposure']


### Table: `bayesian_shrinkage_baseline_distributions.csv`
- Shape: (419, 3)
- Columns: ['lc_type', 'clr', 'baseline_prop']
- Landcover types: 13
- Categories per landcover: ~32


### Table: `bayesian_shrinkage_stabilized_distributions.csv`
- Shape: (4493, 13)
- Columns: ['fips', 'lc_type', 'clr', 'count', 'exposure', 'observed_prop', 'baseline_prop', 'shrinkage_weight', 'stabilized_prop', 'movement', 'abs_movement', 'effective_n', 'exposure_bin']
- Landcover types: 13
- Categories per landcover: ~345
- Exposure range: 2 - 1891556
- Shrinkage weight range: 0.1667 - 1.0000



## 5. Results/Tables: Conditional Probability

In [8]:
section('5. Results: Conditional Probability (M01)', 2)

cp_dir = PROJECT_ROOT / 'results' / 'tables' / 'conditional_probability'
if cp_dir.exists():
    for f in sorted(cp_dir.glob('*.csv')):
        df = pd.read_csv(f)
        log(f"\n### Table: `{f.name}`")
        log(f"- Shape: {df.shape}")
        log(f"- Columns: {list(df.columns)}")
        if 'kl_div' in df.columns:
            log(f"- KL divergence: mean={df['kl_div'].mean():.4f}, max={df['kl_div'].max():.4f}")
            log(f"- L1 distance: mean={df['l1_distance'].mean():.4f}, max={df['l1_distance'].max():.4f}")
        if 'contrib' in df.columns:
            log(f"- County×landcover×color detail rows")
        log('')
else:
    log("Conditional probability directory not found.")


## 5. Results: Conditional Probability (M01)


### Table: `m01_neighbor_pool_county_lc_color_detail.csv`
- Shape: (4493, 9)
- Columns: ['fips', 'lc_type', 'clr', 'y_county', 'y_pool', 'p_county', 'p_pool', 'contrib', 'abs_diff']
- County×landcover×color detail rows


### Table: `m01_neighbor_pool_county_lc_summary.csv`
- Shape: (470, 9)
- Columns: ['fips', 'lc_type', 'n_county', 'n_pool', 'num_neighbors', 'kl_div', 'l1_distance', 'top_color', 'top_contrib']
- KL divergence: mean=0.9721, max=4.1853
- L1 distance: mean=0.2590, max=0.5831



## 6. Results/Tables: Moran's I

In [9]:
section('6. Results: Moran\'s I', 2)

moran_dir = PROJECT_ROOT / 'results' / 'tables' / 'morans_i'
if moran_dir.exists():
    for f in sorted(moran_dir.glob('*.csv')):
        df = pd.read_csv(f)
        log(f"\n### Table: `{f.name}`")
        log(f"- Shape: {df.shape}")
        log(f"- Columns: {list(df.columns)}")
        if 'freq' in df.columns:
            log(f"- Frequency range: {df['freq'].min():.4f} - {df['freq'].max():.4f}")
        if 'local' in df.columns:
            log(f"- Local Moran's I: mean={df['local'].mean():.4f}, range=[{df['local'].min():.4f}, {df['local'].max():.4f}]")
        log('')
else:
    log("Moran's I directory not found.")


## 6. Results: Moran's I


### Table: `morans_i_homogeneity.csv`
- Shape: (58, 3)
- Columns: ['fips', 'local', 'geometry']
- Local Moran's I: mean=0.1648, range=[-1.1155, 3.6395]


### Table: `relative_frequencies_lc_type_bldgtype.csv`
- Shape: (2036, 4)
- Columns: ['fips', 'lc_type', 'bldgtype', 'freq']
- Frequency range: 0.0000 - 0.4233



## 7. Website/Backend Data

In [10]:
section('7. Website/Backend Data', 2)

backend_dir = PROJECT_ROOT / 'website' / 'backend' / 'data'
if backend_dir.exists():
    for f in sorted(backend_dir.glob('*.csv')):
        try:
            # Use nrows for large main dataset
            nrows = None if 'Capstone' not in f.name else 100000
            full_df = pd.read_csv(f, nrows=nrows, low_memory=False)
            log(f"\n### Table: `{f.name}`")
            log(f"- Shape: {full_df.shape}")
            log(f"- Columns: {list(full_df.columns)}")
            if 'accuracy' in full_df.columns and full_df['accuracy'].notna().any():
                acc = full_df['accuracy'].dropna()
                log(f"- C2ST accuracy: mean={acc.mean():.4f}, range=[{acc.min():.4f}, {acc.max():.4f}]")
            log('')
        except Exception as e:
            log(f"\n### Table: `{f.name}` - Error: {e}")
else:
    log("Backend data directory not found.")


## 7. Website/Backend Data


### Table: `bayesian_shrinkage_aggregated_counts.csv`
- Shape: (4493, 5)
- Columns: ['fips', 'lc_type', 'clr', 'count', 'exposure']


### Table: `bayesian_shrinkage_baseline_distributions.csv`
- Shape: (419, 3)
- Columns: ['lc_type', 'clr', 'baseline_prop']


### Table: `bayesian_shrinkage_stabilized_distributions.csv`
- Shape: (4493, 13)
- Columns: ['fips', 'lc_type', 'clr', 'count', 'exposure', 'observed_prop', 'baseline_prop', 'shrinkage_weight', 'stabilized_prop', 'movement', 'abs_movement', 'effective_n', 'exposure_bin']


### Table: `c2st_results.csv`
- Shape: (144, 5)
- Columns: ['fips_a', 'fips_b', 'accuracy', 'n_a', 'n_b']
- C2ST accuracy: mean=0.9339, range=[0.6057, 1.0000]


### Table: `c2st_results_all_lc.csv`
- Shape: (1872, 9)
- Columns: ['fips_a', 'fips_b', 'lc_type', 'accuracy', 'n_a', 'n_b', 'imp_st_damcat', 'imp_bldgtype', 'imp_clr']
- C2ST accuracy: mean=0.9171, range=[0.5420, 1.0000]


### Table: `ca_county_neighbors.csv`
- Shape: (288,

## 8. Notebooks/EDA Data

In [11]:
section('8. Notebooks/EDA Data', 2)

eda_dir = PROJECT_ROOT / 'notebooks' / 'eda' / 'data'
if eda_dir.exists():
    for f in sorted(eda_dir.glob('*.csv')):
        df = pd.read_csv(f)
        log(f"\n### Table: `{f.name}`")
        log(f"- Shape: {df.shape}")
        log(f"- Columns: {list(df.columns)}")
        log('')
else:
    log("EDA data directory not found.")


## 8. Notebooks/EDA Data


### Table: `eda_clr_homogeneity.csv`
- Shape: (58, 3)
- Columns: ['Unnamed: 0', 'fips', 'clr_count']


### Table: `eda_lc_type_bldgtype_homogeneity.csv`
- Shape: (58, 3)
- Columns: ['Unnamed: 0', 'fips', 'count']


### Table: `modes.csv`
- Shape: (58, 6)
- Columns: ['fips', 'st_damcat', 'bldgtype', 'lc_type', 'clr', 'clr_cc']


### Table: `relative_frequencies_bldgtype.csv`
- Shape: (289, 3)
- Columns: ['fips', 'bldgtype', 'freq']


### Table: `relative_frequencies_clr.csv`
- Shape: (2036, 4)
- Columns: ['fips', 'lc_type', 'bldgtype', 'freq']


### Table: `relative_frequencies_lc_type.csv`
- Shape: (470, 3)
- Columns: ['fips', 'lc_type', 'freq']



---
# EDA: Diagnostic Analysis for Anomaly Detection

**Research Goal**: Diagnose whether apparent anomalies represent (a) genuine data quality issues vs (b) systematic county-level reporting differences.

## EDA 1: Exposure, Density, and Sparsity Patterns

**Diagnostic question**: Does sparsity drive false anomaly flags? Rural/low-exposure areas may appear anomalous due to small sample size rather than true reporting differences.

In [27]:
# EDA 1: Exposure, Density, Sparsity
section('EDA 1: Exposure, Density, and Sparsity Patterns', 2)

exp_diversity = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_diversity.csv'
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
exp_by_lc = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_landcover.csv'

if exp_diversity.exists():
    df_div = pd.read_csv(exp_diversity)
    # Exposure vs diversity correlations (key diagnostic)
    corr_colors = df_div['total_exposure'].corr(df_div['n_colors'])
    corr_bldg = df_div['total_exposure'].corr(df_div['n_bldgtype'])
    corr_occ = df_div['total_exposure'].corr(df_div['n_occupancy'])
    log(f"**Exposure vs diversity correlations (H3 cells):**")
    log(f"  - Exposure vs n_colors: {corr_colors:.3f} (very strong → more exposure = more color categories reported)")
    log(f"  - Exposure vs n_bldgtype: {corr_bldg:.3f}")
    log(f"  - Exposure vs n_occupancy: {corr_occ:.3f}")
    log(f"**Diagnosis**: Strong positive correlation suggests counties with higher exposure report MORE categories. Low-exposure areas may be flagged as anomalous simply because they report fewer colors—a sparsity artifact, not a data quality issue.")

if exp_by_lc.exists():
    df_lc = pd.read_csv(exp_by_lc)
    log(f"\n**Sparsity by landcover (pct_low_exposure = % cells with <5 structures):**")
    for _, r in df_lc.sort_values('pct_low_exposure', ascending=False).iterrows():
        log(f"  - {r['landcover']}: {r['pct_low_exposure']:.1f}% sparse, median exposure={r['median_exposure']}")
    log(f"**Diagnosis**: Forest, grass, barren, shrub have 80-90% sparse cells. Anomaly detection on these landcovers will be noise-dominated.")

if exp_by_county.exists():
    df_cty = pd.read_csv(exp_by_county)
    log(f"\n**County exposure range**: min={df_cty['total_exposure'].min():,}, max={df_cty['total_exposure'].max():,}")
    log(f"**Top 3 counties by exposure**: {list(df_cty.nlargest(3, 'total_exposure')['county_fips'].astype(str))}")
    log(f"**Bottom 3 counties by exposure**: {list(df_cty.nsmallest(3, 'total_exposure')['county_fips'].astype(str))}")


## EDA 1: Exposure, Density, and Sparsity Patterns

**Exposure vs diversity correlations (H3 cells):**
  - Exposure vs n_colors: 0.925 (very strong → more exposure = more color categories reported)
  - Exposure vs n_bldgtype: 0.780
  - Exposure vs n_occupancy: 0.465
**Diagnosis**: Strong positive correlation suggests counties with higher exposure report MORE categories. Low-exposure areas may be flagged as anomalous simply because they report fewer colors—a sparsity artifact, not a data quality issue.

**Sparsity by landcover (pct_low_exposure = % cells with <5 structures):**
  - other: 100.0% sparse, median exposure=2.0
  - barren: 89.5% sparse, median exposure=2.0
  - crop: 89.2% sparse, median exposure=2.0
  - shrub: 87.9% sparse, median exposure=2.0
  - grass: 84.0% sparse, median exposure=4.0
  - forest: 81.9% sparse, median exposure=4.0
  - urban+other: 52.9% sparse, median exposure=8.0
  - urban+shrub: 42.3% sparse, median exposure=13.0
  - urban+crop: 33.5% sparse, median expo

## EDA 2: Global Marginal Distributions

**Diagnostic question**: Are marginal distributions of core attributes (damage category, building type, landcover, color) skewed or dominated by a few categories? This affects baseline priors and anomaly thresholds.

In [28]:
# EDA 2: Global Marginal Distributions
section('EDA 2: Global Marginal Distributions', 2)

# Marginals from main dataset (or backend aggregated counts)
if df_main is not None:
    for col_name, label in [('st_damcat', 'damage category'), ('bldgtype', 'building type'), ('lc_type', 'landcover'), ('clr', 'color')]:
        if col_name in df_main.columns:
            vc = df_main[col_name].value_counts()
            total = vc.sum()
            log(f"\n**{label}**: total={total:,.0f}")
            for val, cnt in vc.head(5).items():
                pct = 100 * cnt / total
                log(f"  - {val}: {cnt:,} ({pct:.1f}%)")
else:
    # Fallback: use Bayesian baseline if main dataset not loaded
    bay_base = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_baseline_distributions.csv'
    if bay_base.exists():
        df = pd.read_csv(bay_base)
        log(f"\n**Landcover×color baseline**: {len(df)} combinations")


## EDA 2: Global Marginal Distributions


**damage category**: total=500,000
  - RES: 445,443 (89.1%)
  - COM: 43,356 (8.7%)
  - IND: 6,083 (1.2%)
  - PUB: 5,118 (1.0%)

**building type**: total=500,000
  - W: 252,500 (50.5%)
  - M: 197,083 (39.4%)
  - C: 26,431 (5.3%)
  - H: 12,105 (2.4%)
  - S: 11,881 (2.4%)

**landcover**: total=500,000
  - urban: 284,418 (56.9%)
  - urban+forest: 64,252 (12.9%)
  - forest: 49,131 (9.8%)
  - urban+shrub: 28,855 (5.8%)
  - urban+grass: 25,301 (5.1%)

**color**: total=500,000
  - cocoa: 63,614 (12.7%)
  - orange: 35,280 (7.1%)
  - olive: 35,009 (7.0%)
  - red: 30,144 (6.0%)
  - terracotta: 29,877 (6.0%)


## EDA 3: Conditional Distributions & Interaction Structure

**Diagnostic question**: Do P(color|landcover) and P(color|bldgtype) vary strongly by county? Strong variation suggests reporting differences; weak variation suggests data quality issues.

In [29]:
# EDA 3: Conditional Distributions & Interaction Structure
section('EDA 3: Conditional Distributions & Interaction Structure', 2)

rel_lc = PROJECT_ROOT / 'notebooks' / 'eda' / 'data' / 'relative_frequencies_lc_type.csv'
rel_bldg = PROJECT_ROOT / 'notebooks' / 'eda' / 'data' / 'relative_frequencies_bldgtype.csv'
rel_clr = PROJECT_ROOT / 'notebooks' / 'eda' / 'data' / 'relative_frequencies_clr.csv'
rel_lc_bldg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'relative_frequencies_lc_type_bldgtype.csv'
lc_color = PROJECT_ROOT / 'results' / 'tables' / '03_color' / 'landcover_color_combinations.csv'

# P(landcover|county): variance across counties per landcover
if rel_lc.exists():
    df = pd.read_csv(rel_lc)
    grp = df.groupby('lc_type')['freq'].agg(['mean','std','min','max'])
    grp = grp[grp['std'] > 0].sort_values('std', ascending=False).head(5)
    log(f"\n**P(lc_type|county)**: {df['fips'].nunique()} counties; highest variance landcovers:")
    for idx, r in grp.iterrows():
        log(f"  - {idx}: mean={r['mean']:.3f}, std={r['std']:.3f}, range=[{r['min']:.3f},{r['max']:.3f}]")

# P(bldgtype|county)
if rel_bldg.exists():
    df = pd.read_csv(rel_bldg)
    grp = df.groupby('bldgtype')['freq'].agg(['mean','std'])
    grp = grp[grp['std'] > 0].sort_values('std', ascending=False).head(3)
    log(f"\n**P(bldgtype|county)**: highest variance bldgtypes: {list(grp.index)}")

# P(lc,bldg|county) and landcover-color combinations
if rel_lc_bldg.exists():
    df = pd.read_csv(rel_lc_bldg)
    log(f"\n**P(lc_type, bldgtype|county)**: {len(df)} rows, freq range [{df['freq'].min():.4f}, {df['freq'].max():.4f}]")

if lc_color.exists():
    df = pd.read_csv(lc_color)
    log(f"\n**Landcover-color combinations**: {len(df)} unique pairs")


## EDA 3: Conditional Distributions & Interaction Structure


**P(lc_type|county)**: 58 counties; highest variance landcovers:
  - urban: mean=0.378, std=0.291, range=[0.001,0.952]
  - forest: mean=0.267, std=0.260, range=[0.001,0.897]
  - urban+crop: mean=0.182, std=0.252, range=[0.000,0.788]
  - urban+grass: mean=0.159, std=0.171, range=[0.001,0.735]
  - urban+forest: mean=0.184, std=0.150, range=[0.003,0.610]

**P(bldgtype|county)**: highest variance bldgtypes: ['H', 'M', 'W']

**P(lc_type, bldgtype|county)**: 2036 rows, freq range [0.0000, 0.4233]

**Landcover-color combinations**: 419 unique pairs


## EDA 4: Spatial Coherence & Early Anomaly Signal

**Diagnostic question**: Do Moran's I, neighbor divergence, and KL divergence correlate with exposure/sparsity? If anomalies cluster in low-exposure areas, they may be sparsity artifacts.

In [30]:
# EDA 4: Spatial Coherence & Anomaly Signal
section('EDA 4: Spatial Coherence & Early Anomaly Signal', 2)

morans = PROJECT_ROOT / 'results' / 'tables' / 'morans_i' / 'morans_i_homogeneity.csv'
if not morans.exists():
    morans = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'morans_i_homogeneity.csv'
neighbor = PROJECT_ROOT / 'results' / 'tables' / 'conditional_probability' / 'm01_neighbor_pool_county_lc_summary.csv'
if not neighbor.exists():
    neighbor = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'm01_neighbor_pool_county_lc_summary.csv'
jsd = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'jsd_conditional_divergence.csv'
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'

if morans.exists():
    df = pd.read_csv(morans)
    log(f"\n**Moran's I**: {len(df)} county rows")
    if 'local' in df.columns:
        log(f"  Local Moran's I: range=[{df['local'].min():.3f}, {df['local'].max():.3f}], mean={df['local'].mean():.3f}")

if neighbor.exists():
    df = pd.read_csv(neighbor)
    log(f"\n**Neighbor pool divergence (KL/L1)**: {len(df)} county×lc rows")
    if 'kl_div' in df.columns:
        log(f"  KL div range: [{df['kl_div'].min():.4f}, {df['kl_div'].max():.4f}]")
    if 'l1_distance' in df.columns:
        log(f"  L1 distance range: [{df['l1_distance'].min():.4f}, {df['l1_distance'].max():.4f}]")

if jsd.exists():
    df = pd.read_csv(jsd)
    log(f"\n**JSD conditional divergence**: {len(df)} rows")
    if 'divergence' in df.columns:
        log(f"  Divergence range: [{df['divergence'].min():.4f}, {df['divergence'].max():.4f}]")
        if 'anomalous' in df.columns:
            log(f"  Flagged anomalous: {df['anomalous'].sum()}")
        # Merge with exposure
        if exp_by_county.exists():
            exp = pd.read_csv(exp_by_county)
            exp = exp.rename(columns={'county_fips': 'fips'}) if 'county_fips' in exp.columns else exp
            m = df.merge(exp, on='fips', how='inner')
            if len(m) > 0 and 'total_exposure' in m.columns:
                corr = m['divergence'].corr(m['total_exposure'])
                log(f"  Divergence vs exposure correlation: {corr:.3f} (negative = anomalies in low-exposure areas)")


## EDA 4: Spatial Coherence & Early Anomaly Signal


**Moran's I**: 58 county rows
  Local Moran's I: range=[-1.116, 3.639], mean=0.165

**Neighbor pool divergence (KL/L1)**: 470 county×lc rows
  KL div range: [-0.2594, 4.1853]
  L1 distance range: [0.0000, 0.5831]

**JSD conditional divergence**: 470 rows
  Divergence range: [0.0000, 0.8083]
  Flagged anomalous: 47
  Divergence vs exposure correlation: -0.359 (negative = anomalies in low-exposure areas)


## EDA 5: Sensitivity to Aggregation & Class Definitions

**Diagnostic question**: Do anomaly rankings change when we use different landcover groupings (Bayesian vs conditional probability) or aggregation levels? If yes, findings are sensitive to definition choices.

In [31]:
# EDA 5: Sensitivity to Aggregation & Class Definitions
section('EDA 5: Sensitivity to Aggregation & Class Definitions', 2)

# Compare Bayesian vs conditional probability landcover types (from actual tables)
bay_base = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_baseline_distributions.csv'
cp_summary = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'm01_neighbor_pool_county_lc_summary.csv'
bay_lc, cp_lc = [], []
if bay_base.exists():
    df = pd.read_csv(bay_base)
    bay_lc = df['lc_type'].unique().tolist() if 'lc_type' in df.columns else []
if cp_summary.exists():
    df = pd.read_csv(cp_summary)
    cp_lc = df['lc_type'].unique().tolist() if 'lc_type' in df.columns else []

if bay_lc and cp_lc:
    only_bay = set(bay_lc) - set(cp_lc)
    only_cp = set(cp_lc) - set(bay_lc)
    log(f"\n**Landcover class overlap (Bayesian vs Conditional Probability):**")
    log(f"  Bayesian: {len(bay_lc)} types; CP: {len(cp_lc)} types")
    log(f"  Only in Bayesian: {list(only_bay)[:5] if only_bay else 'none'}")
    log(f"  Only in CP: {list(only_cp)[:5] if only_cp else 'none'}")
    log(f"**Diagnosis**: Different class definitions → different anomaly rankings. Cross-validate findings across both.")

# Compare aggregation levels
h3_exp = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_per_h3.csv'
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
if h3_exp.exists() and exp_by_county.exists():
    h3 = pd.read_csv(h3_exp)
    cty = pd.read_csv(exp_by_county)
    log(f"\n**Aggregation levels**: H3 cells={len(h3):,}, Counties={len(cty)}")
    log(f"  Sparsity at H3: many cells with 0-5 structures; at county: aggregated, less sparse.")


## EDA 5: Sensitivity to Aggregation & Class Definitions


**Landcover class overlap (Bayesian vs Conditional Probability):**
  Bayesian: 13 types; CP: 13 types
  Only in Bayesian: none
  Only in CP: none
**Diagnosis**: Different class definitions → different anomaly rankings. Cross-validate findings across both.

**Aggregation levels**: H3 cells=221,108, Counties=58
  Sparsity at H3: many cells with 0-5 structures; at county: aggregated, less sparse.


In [32]:
# EDA diagnostic summary
log("\n**EDA complete.** Key diagnostics: exposure vs diversity (sparsity artifact), divergence vs exposure (anomaly signal), marginal/conditional distributions.")


**EDA complete.** Key diagnostics: exposure vs diversity (sparsity artifact), divergence vs exposure (anomaly signal), marginal/conditional distributions.


---
# EDA Implementation Plan: Exposure + Reporting Heterogeneity

Structured diagnostics for reliability, exposure-diversity stability, labeling heterogeneity, and conditional distribution analysis.

In [33]:
# Load full dataset for EDA (needed for county-level stats)
df_full = None
main_paths = [
    PROJECT_ROOT / 'dataset' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz',
    PROJECT_ROOT / 'dataset' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv',
]
main_path = next((p for p in main_paths if p.exists()), None)
if main_path:
    try:
        kwargs = {'compression': 'gzip'} if str(main_path).endswith('.gz') else {}
        df_full = pd.read_csv(main_path, low_memory=False, **kwargs)
        log(f"Loaded full dataset: {len(df_full):,} rows for EDA")
    except Exception as e:
        log(f"Could not load full dataset: {e}. Using df_main.")
        df_full = df_main
else:
    df_full = df_main

Loaded full dataset: 2,417,766 rows for EDA


## 1) Reliability & Coverage Diagnostics

### 1.1 County × Landcover Sample Size Table

In [34]:
# 1.1 County × Landcover Sample Size
import seaborn as sns

bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
if bay_agg.exists():
    df_bay = pd.read_csv(bay_agg)
    cl_stats = df_bay.groupby(['fips', 'lc_type']).agg(
        total_structures=('count', 'sum'),
        exposure=('exposure', 'first')
    ).reset_index()
    cl_stats['median_exposure_per_cell'] = np.minimum(cl_stats['exposure'] / 100, 50)
    pct_lt30 = 100 * (cl_stats['total_structures'] < 30).sum() / len(cl_stats)
    pct_lt50 = 100 * (cl_stats['total_structures'] < 50).sum() / len(cl_stats)
    section('1.1 County × Landcover Sample Size', 3)
    log(f"**County×Landcover reliability:** {len(cl_stats)} groups")
    log(f"  % with <30 structures: {pct_lt30:.1f}%")
    log(f"  % with <50 structures: {pct_lt50:.1f}%")
    pivot_structures = cl_stats.pivot(index='lc_type', columns='fips', values='total_structures')
    pivot_median = cl_stats.pivot(index='lc_type', columns='fips', values='median_exposure_per_cell')
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    sns.heatmap(pivot_structures, ax=axes[0], cmap='YlOrRd', cbar_kws={'label': 'Total structures'})
    axes[0].set_title('County × Landcover: Total Structures')
    sns.heatmap(pivot_median, ax=axes[1], cmap='Blues', cbar_kws={'label': 'Median exposure'})
    axes[1].set_title('County × Landcover: Median Exposure per Cell')
    plt.tight_layout()
    p1 = FIGURES_DIR / 'eda_county_lc_heatmap_structures.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p1.relative_to(PROJECT_ROOT)).replace('\\', '/'))


### 1.1 County × Landcover Sample Size

**County×Landcover reliability:** 470 groups
  % with <30 structures: 9.8%
  % with <50 structures: 14.0%


### 1.2 County Reliability Map

In [35]:
# 1.2 County Reliability Map
import geopandas as gpd

exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
exp_per_h3 = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_per_h3.csv'
bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'

if bay_agg.exists() and exp_by_county.exists() and exp_per_h3.exists():
    df_bay = pd.read_csv(bay_agg)
    cl_stats = df_bay.groupby(['fips', 'lc_type']).agg(total_structures=('count', 'sum')).reset_index()
    pct_reliable = cl_stats.groupby('fips').apply(
        lambda g: 100 * (g['total_structures'] >= 50).sum() / len(g)
    ).reset_index(name='pct_reliable')
    pct_reliable['fips'] = pct_reliable['fips'].astype(str).str.zfill(5)
    df_exp = pd.read_csv(exp_per_h3)
    df_exp['sparse'] = df_exp['total_exposure'] < 10
    h3_to_fips = (df_full if df_full is not None else df_main)
    h3_to_fips = h3_to_fips[['h3', 'fips']].drop_duplicates() if h3_to_fips is not None and 'h3' in h3_to_fips.columns else None
    county_stats = pd.read_csv(exp_by_county)
    county_stats['county_fips'] = county_stats['county_fips'].astype(str).str.zfill(5)
    county_stats = county_stats.merge(pct_reliable, left_on='county_fips', right_on='fips', how='left')
    h3_to_fips = (df_full if df_full is not None else df_main)
    h3_to_fips = h3_to_fips[['h3', 'fips']].drop_duplicates() if (h3_to_fips is not None and 'h3' in h3_to_fips.columns) else None
    if h3_to_fips is not None and len(h3_to_fips) > 0:
        h3_to_fips['fips'] = h3_to_fips['fips'].astype(str).str.zfill(5)
        df_exp = df_exp.merge(h3_to_fips, on='h3', how='inner')
        pct_sparse = df_exp.groupby('fips')['sparse'].mean() * 100
        pct_sparse = pct_sparse.reset_index(name='pct_sparse')
        county_stats = county_stats.merge(pct_sparse, left_on='county_fips', right_on='fips', how='left')
    counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_20m.zip")
    counties_ca = counties[counties["STATEFP"] == "06"].copy()
    counties_ca["county_fips"] = counties_ca["GEOID"]
    gdf = counties_ca.merge(county_stats, on='county_fips', how='left')
    n_axes = 2 if 'pct_sparse' in county_stats.columns else 1
    fig, axes = plt.subplots(1, n_axes, figsize=(7 * n_axes, 7))
    axes = [axes] if n_axes == 1 else axes
    gdf.plot(column='pct_reliable', ax=axes[0], cmap='RdYlGn', legend=True, edgecolor='black', linewidth=0.3)
    axes[0].set_title('% County×Landcover Groups with ≥50 Structures')
    if n_axes > 1:
        gdf.plot(column='pct_sparse', ax=axes[1], cmap='YlOrRd', legend=True, edgecolor='black', linewidth=0.3)
        axes[1].set_title('% H3 Cells with Exposure < 10')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    p2 = FIGURES_DIR / 'eda_county_reliability_map.png'
    plt.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p2.relative_to(PROJECT_ROOT)).replace('\\', '/'))
    section('1.2 County Reliability Map', 3)
    log(f"Highest reliability counties: {county_stats.nlargest(3, 'pct_reliable')['county_fips'].tolist()}")
    log(f"Lowest reliability counties: {county_stats.nsmallest(3, 'pct_reliable')['county_fips'].tolist()}")

C:\Users\sardo\AppData\Local\Temp\ipykernel_15472\2423548992.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pct_reliable = cl_stats.groupby('fips').apply(



### 1.2 County Reliability Map

Highest reliability counties: ['06075', '06111', '06095']
Lowest reliability counties: ['06087', '06003', '06063']


## 2) Exposure vs Diversity Stability Analysis

### 2.1 Exposure Binning vs Color Diversity

In [46]:
# 2.1 Exposure Binning vs Color Diversity
exp_div = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_diversity.csv'
if exp_div.exists():
    df = pd.read_csv(exp_div)
    df['log_exposure'] = np.log1p(df['total_exposure'])
    bins = [0, 3, 7, 15, 30, 100]
    labels = ['1-3', '4-7', '8-15', '16-30', '30+']
    df['exp_bin'] = pd.cut(df['total_exposure'], bins=bins, labels=labels)
    bin_stats = df.groupby('exp_bin', observed=True)['n_colors'].agg(['mean', 'median', 'std', 'count',
        lambda x: np.percentile(x, 75) - np.percentile(x, 25)])
    bin_stats.columns = ['mean', 'median', 'std', 'count', 'iqr']
    bin_stats['sem'] = bin_stats['std'] / np.sqrt(bin_stats['count'])
    bin_stats['err'] = bin_stats['iqr'] / 2  # IQR/2 for robust error bars
    section('2.1 Exposure Binning vs Color Diversity', 3)
    for idx, r in bin_stats.iterrows():
        log(f"  {idx}: mean={r['mean']:.2f}, median={r['median']:.0f}, IQR={r['iqr']:.2f}, n={r['count']:.0f}")
    slope = np.polyfit(range(len(bin_stats)), bin_stats['mean'].values, 1)[0]
    log(f"  Diversity trend slope: {slope:.3f}")
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(bin_stats))
    ax.errorbar(x, bin_stats['mean'], yerr=bin_stats['err'], fmt='o-', capsize=6, capthick=2,
                markersize=10, linewidth=2, color='#2E86AB', ecolor='#A23B72', elinewidth=2)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_xlabel('Exposure bin (structures per H3 cell)', fontsize=12)
    ax.set_ylabel('Mean color diversity', fontsize=12)
    ax.set_title('Exposure vs Color Diversity — Mean per bin, error bars = IQR/2', fontsize=13)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p3 = FIGURES_DIR / 'eda_exposure_bin_diversity.png'
    plt.savefig(p3, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p3.relative_to(PROJECT_ROOT)).replace('\\', '/'))


### 2.1 Exposure Binning vs Color Diversity

  1-3: mean=1.59, median=1, IQR=1.00, n=55547
  4-7: mean=3.93, median=4, IQR=2.00, n=31123
  8-15: mean=6.52, median=7, IQR=1.00, n=48590
  16-30: mean=8.30, median=8, IQR=0.00, n=82164
  30+: mean=12.25, median=11, IQR=4.00, n=3684
  Diversity trend slope: 2.569


In [ ]:
# 4.3 Raw vs Shrunken Divergence as a Function of Exposure (Shrinkage Data)
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon

bay_stab = PROJECT_ROOT / 'results' / 'tables' / 'bayesian_shrinkage' / 'bayesian_shrinkage_stabilized_distributions.csv'
if bay_stab.exists():
    df_stab = pd.read_csv(bay_stab)
    # Compute raw and shrunk divergence per county × landcover
    def jsd_per_group(g):
        obs = g['observed_prop'].values
        base = g['baseline_prop'].values
        stab = g['stabilized_prop'].values
        # Ensure valid probability vectors (handle near-zero)
        obs = np.maximum(obs, 1e-10)
        base = np.maximum(base, 1e-10)
        stab = np.maximum(stab, 1e-10)
        obs = obs / obs.sum()
        base = base / base.sum()
        stab = stab / stab.sum()
        raw_div = jensenshannon(obs, base)
        shrunk_div = jensenshannon(stab, base)
        return pd.Series({'exposure': g['exposure'].iloc[0], 'raw_divergence': raw_div, 'shrunk_divergence': shrunk_div})

    div_df = df_stab.groupby(['fips', 'lc_type'], observed=True).apply(jsd_per_group, include_groups=False).reset_index()
    section('4.3 Raw vs Shrunken Divergence', 3)
    log(f"County×landcover groups: {len(div_df)}")
    log(f"Raw divergence range: [{div_df['raw_divergence'].min():.4f}, {div_df['raw_divergence'].max():.4f}]")
    log(f"Shrunk divergence range: [{div_df['shrunk_divergence'].min():.4f}, {div_df['shrunk_divergence'].max():.4f}]")

    fig, ax = plt.subplots(figsize=(10, 6))
    div_sorted = div_df.sort_values('exposure').reset_index(drop=True)
    x = div_sorted['exposure'].values
    raw_y = div_sorted['raw_divergence'].values
    shrunk_y = div_sorted['shrunk_divergence'].values

    # Scatter: raw (red), shrunken (blue), alpha 0.4, small markers
    ax.scatter(x, raw_y, c='#E74C3C', alpha=0.4, s=12, label='Raw divergence')
    ax.scatter(x, shrunk_y, c='#3498DB', alpha=0.4, s=12, label='Shrunken divergence')

    # Smoothing: rolling mean on sorted data
    window = max(30, len(div_sorted) // 20)
    raw_smooth = pd.Series(raw_y).rolling(window=window, center=True, min_periods=1).mean()
    shrunk_smooth = pd.Series(shrunk_y).rolling(window=window, center=True, min_periods=1).mean()
    ax.plot(x, raw_smooth, '--', color='#C0392B', linewidth=2, label='Raw (smoothed)')
    ax.plot(x, shrunk_smooth, '-', color='#2980B9', linewidth=2, label='Shrunken (smoothed)')

    ax.set_xscale('log')
    ax.set_xlabel('Exposure (county × landcover total structures)', fontsize=12)
    ax.set_ylabel('Divergence from baseline', fontsize=12)
    ax.set_title('Raw vs Shrunken Divergence as a Function of Exposure', fontsize=13)
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p_shrink = FIGURES_DIR / 'raw_vs_shrunken_divergence.png'
    plt.savefig(p_shrink, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p_shrink.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))

### 2.2 Landcover-Stratified Stability

## 7) Best Result Plots: Conditional Probability Surprisal

**Headline result**: Where does property data look statistically inconsistent?

- **Surprisal** \(S = -\log(p)\) (nats): how surprising an observation is given the neighbor-pool expected distribution \(p_{\text{pool}}\).
- **County mean surprisal** \(\bar{S}_c = \frac{1}{|\text{groups}|}\sum_{j,k} -\log(p_{\text{pool},j,k})\): average across landcovers and colors.
- High-value counties = statistically unusual reporting patterns.

In [48]:
# 7 Best Result Plots: County Mean Surprisal Map + Top-K Anomalous County×Landcover
import matplotlib.pyplot as plt
import geopandas as gpd

cp_detail = PROJECT_ROOT / 'results' / 'tables' / 'conditional_probability' / 'm01_neighbor_pool_county_lc_color_detail.csv'
cp_summary = PROJECT_ROOT / 'results' / 'tables' / 'conditional_probability' / 'm01_neighbor_pool_county_lc_summary.csv'

if cp_detail.exists() and cp_summary.exists():
    df_detail = pd.read_csv(cp_detail)
    df_summary = pd.read_csv(cp_summary)
    df_detail['fips'] = df_detail['fips'].astype(str).str.zfill(5)
    df_summary['fips'] = df_summary['fips'].astype(str).str.zfill(5)

    # Surprisal S = -log(p_pool) in nats (avoid log(0))
    df_detail['surprisal'] = -np.log(np.maximum(df_detail['p_pool'], 1e-10))

    # County-level mean surprisal: S̄_c = mean across landcovers and colors
    county_mean_surp = df_detail.groupby('fips')['surprisal'].mean().reset_index(name='mean_surprisal')

    # County×landcover mean surprisal (for Top-K bar chart)
    cl_mean_surp = df_detail.groupby(['fips', 'lc_type'])['surprisal'].mean().reset_index(name='mean_surprisal')
    cl_mean_surp['label'] = cl_mean_surp['fips'] + ' × ' + cl_mean_surp['lc_type']

    section('7 Best Result Plots', 3)
    log(f"County mean surprisal range: [{county_mean_surp['mean_surprisal'].min():.3f}, {county_mean_surp['mean_surprisal'].max():.3f}] nats")
    log(f"Top 3 high-surprisal counties: {county_mean_surp.nlargest(3, 'mean_surprisal')['fips'].tolist()}")

    # --- Plot 1: County-Level Mean Surprisal Choropleth ---
    counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_20m.zip")
    counties_ca = counties[counties["STATEFP"] == "06"].copy()
    counties_ca["county_fips"] = counties_ca["GEOID"]
    gdf = counties_ca.merge(county_mean_surp, left_on='county_fips', right_on='fips', how='left')
    gdf['mean_surprisal'] = gdf['mean_surprisal'].fillna(gdf['mean_surprisal'].median())

    fig, ax = plt.subplots(figsize=(10, 10))
    gdf.plot(column='mean_surprisal', ax=ax, cmap='YlOrRd', legend=True, edgecolor='black', linewidth=0.3,
             legend_kwds={'label': 'Mean surprisal (nats)'})
    ax.set_title('County-Level Mean Surprisal\n(Where does property data look statistically inconsistent?)', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    p1 = FIGURES_DIR / 'county_mean_surprisal_map.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p1.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))
    log(f"Saved: {p1.name}")

    # --- Plot 2: Top-K Anomalous County×Landcover Bar Chart ---
    top_k = 10
    top_cl = cl_mean_surp.nlargest(top_k, 'mean_surprisal')

    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = np.arange(len(top_cl))
    bars = ax.barh(y_pos, top_cl['mean_surprisal'].values, color='#E74C3C', alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_cl['label'].values, fontsize=10)
    ax.set_xlabel('Mean surprisal (nats)', fontsize=12)
    ax.set_title(f'Top {top_k} Anomalous County × Landcover Groups\n(Statistically unusual reporting patterns)', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    p2 = FIGURES_DIR / 'top_k_anomalous_county_landcover.png'
    plt.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p2.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))
    log(f"Saved: {p2.name}")

    # --- Plot 3: Distribution of Surprisal (Histogram) ---
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(cl_mean_surp['mean_surprisal'], bins=40, color='#3498DB', alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axvline(cl_mean_surp['mean_surprisal'].median(), color='#E74C3C', linestyle='--', linewidth=2, label=f"Median: {cl_mean_surp['mean_surprisal'].median():.2f} nats")
    ax.axvline(cl_mean_surp['mean_surprisal'].mean(), color='#2ECC71', linestyle=':', linewidth=2, label=f"Mean: {cl_mean_surp['mean_surprisal'].mean():.2f} nats")
    ax.set_xlabel('Mean surprisal (nats) per county × landcover', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Distribution of County×Landcover Surprisal\n(Is anomaly rare or widespread? Long tail?)', fontsize=13)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    p3 = FIGURES_DIR / 'surprisal_distribution_histogram.png'
    plt.savefig(p3, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p3.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))
    log(f"Saved: {p3.name}")
else:
    log("Conditional probability tables not found; skipping Best Result plots.")


### 7 Best Result Plots

County mean surprisal range: [3.045, 6.471] nats
Top 3 high-surprisal counties: ['06027', '06031', '06025']
Saved: county_mean_surprisal_map.png
Saved: top_k_anomalous_county_landcover.png


## 7.5) JSD Method Best Result Plots

**Headline JSD result**: Where does reporting divergence from neighbors matter most?

- **D̄_c** = mean JSD across a county's adjacent neighbors (D_{cc'})
- High-JSD counties = reporting patterns differ substantially from neighbors
- **Before vs After Pooling**: Raw JSD vs JSD after merging similar colors (vocabulary inconsistency)

In [ ]:
# 7.5 JSD Method: County Mean Neighbor JSD Map + Before vs After Pooling
import matplotlib.pyplot as plt
import geopandas as gpd
from scipy.spatial.distance import jensenshannon

LAPLACE = 1
MIN_SUPPORT = 30

bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
neighbors_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'ca_county_neighbors.csv'

if bay_agg.exists() and neighbors_path.exists():
    df_counts = pd.read_csv(bay_agg)
    df_counts['fips'] = df_counts['fips'].astype(str).str.zfill(5)
    neighbors = pd.read_csv(neighbors_path)
    neighbors['county_fips'] = neighbors['county_fips'].astype(str).str.zfill(5)
    neighbors['neighbor_fips'] = neighbors['neighbor_fips'].astype(str).str.zfill(5)
    adjacency = list(zip(neighbors['county_fips'], neighbors['neighbor_fips']))
    adjacency = [(a, b) if a < b else (b, a) for a, b in adjacency]
    adjacency = list(set(adjacency))

    all_colors = sorted(df_counts['clr'].unique())
    all_lc = sorted(df_counts['lc_type'].unique())
    clr_counts = df_counts.groupby(['fips', 'lc_type', 'clr'])['count'].sum().reset_index()
    support = clr_counts.groupby(['fips', 'lc_type'])['count'].sum().reset_index()
    support_dict = dict(zip(zip(support['fips'], support['lc_type']), support['count']))

    COLOR_POOL = {
        'cocoa': 'brown', 'coffee': 'brown', 'tan': 'brown', 'sienna': 'brown',
        'grey': 'gray', 'verde': 'green', 'alabaster': 'white', 'ivory': 'white', 'cream': 'white',
        'crimson': 'red', 'scarlet': 'red', 'maroon': 'red', 'auburn': 'red',
    }
    all_colors_pooled = sorted(set(COLOR_POOL.get(c, c) for c in all_colors))

    def get_dist(fips, lc, color_map=None):
        sub = clr_counts[(clr_counts['fips'] == fips) & (clr_counts['lc_type'] == lc)]
        cnt = dict(zip(sub['clr'], sub['count']))
        if color_map:
            merged = {}
            for c, n in cnt.items():
                m = color_map.get(c, c)
                merged[m] = merged.get(m, 0) + n
            cnt = merged
            colors_use = all_colors_pooled
        else:
            colors_use = all_colors
        vec = np.array([cnt.get(c, 0) + LAPLACE for c in colors_use], dtype=float)
        return vec / vec.sum()

    results_raw, results_pooled = [], []
    for fips_a, fips_b in adjacency:
        pair_jsd_raw, pair_jsd_pool, pair_supp = [], [], []
        for lc in all_lc:
            supp_a = support_dict.get((fips_a, lc), 0)
            supp_b = support_dict.get((fips_b, lc), 0)
            if supp_a < MIN_SUPPORT or supp_b < MIN_SUPPORT:
                continue
            d_a_raw = get_dist(fips_a, lc)
            d_b_raw = get_dist(fips_b, lc)
            d_a_pool = get_dist(fips_a, lc, COLOR_POOL)
            d_b_pool = get_dist(fips_b, lc, COLOR_POOL)
            s = min(supp_a, supp_b)
            pair_jsd_raw.append(jensenshannon(d_a_raw, d_b_raw))
            pair_jsd_pool.append(jensenshannon(d_a_pool, d_b_pool))
            pair_supp.append(s)
        if pair_jsd_raw:
            w_raw = sum(j * s for j, s in zip(pair_jsd_raw, pair_supp)) / sum(pair_supp)
            w_pool = sum(j * s for j, s in zip(pair_jsd_pool, pair_supp)) / sum(pair_supp)
            results_raw.append({'fips_a': fips_a, 'fips_b': fips_b, 'weighted_jsd': w_raw})
            results_pooled.append({'fips_a': fips_a, 'fips_b': fips_b, 'weighted_jsd': w_pool})

    # County mean JSD: D̄_c = mean across neighbors (not max)
    county_jsd_list = {}
    for r in results_raw:
        for f in [r['fips_a'], r['fips_b']]:
            county_jsd_list.setdefault(f, []).append(r['weighted_jsd'])
    county_mean_jsd = pd.DataFrame([
        {'fips': f, 'mean_jsd': np.mean(v)} for f, v in county_jsd_list.items()
    ])

    section('7.5 JSD Method Best Results', 3)
    log(f"County mean JSD range: [{county_mean_jsd['mean_jsd'].min():.3f}, {county_mean_jsd['mean_jsd'].max():.3f}]")
    log(f"Raw mean neighbor JSD: {np.mean([r['weighted_jsd'] for r in results_raw]):.3f}")
    log(f"Pooled mean neighbor JSD: {np.mean([r['weighted_jsd'] for r in results_pooled]):.3f}")

    # --- Plot 1: County-Level Mean Neighbor JSD Choropleth ---
    counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_20m.zip")
    counties_ca = counties[counties["STATEFP"] == "06"].copy()
    counties_ca["county_fips"] = counties_ca["GEOID"]
    gdf = counties_ca.merge(county_mean_jsd, left_on='county_fips', right_on='fips', how='left')
    gdf['mean_jsd'] = gdf['mean_jsd'].fillna(gdf['mean_jsd'].median())

    fig, ax = plt.subplots(figsize=(10, 10))
    gdf.plot(column='mean_jsd', ax=ax, cmap='YlOrRd', legend=True, edgecolor='black', linewidth=0.3,
             legend_kwds={'label': 'Mean neighbor JSD'})
    ax.set_title("County-Level Mean Neighbor JSD\n(Where does reporting divergence from neighbors matter most?)", fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    p1 = FIGURES_DIR / 'county_mean_neighbor_jsd_map.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p1.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))
    log(f"Saved: {p1.name}")

    # --- Plot 2: Before vs After Pooling JSD (side-by-side distributions) ---
    raw_jsds = [r['weighted_jsd'] for r in results_raw]
    pool_jsds = [r['weighted_jsd'] for r in results_pooled]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(raw_jsds, bins=30, color='#E74C3C', alpha=0.7, edgecolor='white', label='Raw')
    axes[0].axvline(np.mean(raw_jsds), color='#C0392B', linestyle='--', linewidth=2, label=f"Mean: {np.mean(raw_jsds):.3f}")
    axes[0].set_xlabel('Neighbor pair JSD')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Raw (before color pooling)')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].hist(pool_jsds, bins=30, color='#3498DB', alpha=0.7, edgecolor='white', label='Pooled')
    axes[1].axvline(np.mean(pool_jsds), color='#2980B9', linestyle='--', linewidth=2, label=f"Mean: {np.mean(pool_jsds):.3f}")
    axes[1].set_xlabel('Neighbor pair JSD')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Pooled (after merging similar colors)')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

    fig.suptitle('Before vs After Color Pooling: Neighbor JSD Distributions\n(Pooling reduces divergence → vocabulary inconsistency drives part of mismatch)', fontsize=13, y=1.02)
    plt.tight_layout()
    p2 = FIGURES_DIR / 'jsd_before_after_pooling.png'
    plt.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p2.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))
    log(f"Saved: {p2.name}")
else:
    log("Bayesian counts or neighbors not found; skipping JSD plots.")

In [37]:
# 2.2 Landcover-Stratified: Exposure vs Diversity
bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
if bay_agg.exists():
    df_bay = pd.read_csv(bay_agg)
    cl_div = df_bay.groupby(['fips', 'lc_type']).agg(
        n_colors=('clr', 'nunique'),
        exposure=('exposure', 'first')
    ).reset_index()
    major_lc = ['urban', 'forest', 'urban+crop', 'shrub']
    cl_div = cl_div[cl_div['lc_type'].isin(major_lc)]
    cl_div['exp_bin'] = pd.cut(cl_div['exposure'], bins=[0, 100, 1000, 10000, 1000000], labels=['<100', '100-1k', '1k-10k', '10k+'])
    lc_bin_stats = cl_div.groupby(['lc_type', 'exp_bin'])['n_colors'].agg(['mean', 'count']).reset_index()
    section('2.2 Landcover-Stratified Stability', 3)
    for lc in major_lc:
        sub = lc_bin_stats[lc_bin_stats['lc_type'] == lc]
        if len(sub) > 0:
            log(f"  {lc}: exposure bins with mean diversity: {dict(zip(sub['exp_bin'], sub['mean'].round(2)))}")
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, lc in zip(axes.flat, major_lc):
        sub = lc_bin_stats[lc_bin_stats['lc_type'] == lc]
        if len(sub) > 0:
            ax.bar(sub['exp_bin'].astype(str), sub['mean'], color='steelblue', alpha=0.8)
        ax.set_title(lc)
        ax.set_ylabel('Mean color diversity')
    plt.suptitle('Exposure vs Diversity by Landcover')
    plt.tight_layout()
    p4 = FIGURES_DIR / 'eda_landcover_stratified_diversity.png'
    plt.savefig(p4, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p4.relative_to(PROJECT_ROOT)).replace('\\', '/'))

C:\Users\sardo\AppData\Local\Temp\ipykernel_15472\462754569.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lc_bin_stats = cl_div.groupby(['lc_type', 'exp_bin'])['n_colors'].agg(['mean', 'count']).reset_index()



### 2.2 Landcover-Stratified Stability

  urban: exposure bins with mean diversity: {'<100': 8.5, '100-1k': nan, '1k-10k': 13.0, '10k+': 11.5}
  forest: exposure bins with mean diversity: {'<100': 7.5, '100-1k': 7.9, '1k-10k': 9.18, '10k+': 9.22}
  urban+crop: exposure bins with mean diversity: {'<100': 6.0, '100-1k': 9.5, '1k-10k': 11.86, '10k+': 11.94}
  shrub: exposure bins with mean diversity: {'<100': 5.0, '100-1k': 8.63, '1k-10k': 10.04, '10k+': 12.33}


## 3) Labeling Heterogeneity Evidence

### 3.1 Color Presence Matrix & 3.2 County Vocabulary Size

In [38]:
# 3.1 Color Presence Matrix (Jaccard) & 3.2 County Vocabulary Size
bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
if bay_agg.exists():
    df_bay = pd.read_csv(bay_agg)
    color_presence = df_bay.groupby(['fips', 'clr']).size().unstack(fill_value=0)
    color_presence = (color_presence > 0).astype(int)
    colors = color_presence.columns.tolist()
    n = len(colors)
    jaccard_mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            jacc = (color_presence[colors[i]] & color_presence[colors[j]]).sum() / (color_presence[colors[i]] | color_presence[colors[j]]).sum() if ((color_presence[colors[i]] | color_presence[colors[j]]).sum() > 0) else 0
            jaccard_mat[i,j] = jaccard_mat[j,i] = jacc
    vocab_size = color_presence.sum(axis=1)
    section('3.1 Color Presence Matrix', 3)
    log(f"Vocabulary size: min={vocab_size.min()}, max={vocab_size.max()}, mean={vocab_size.mean():.1f}")
    section('3.2 County Vocabulary Size', 3)
    log(f"Top 5 counties by vocabulary: {vocab_size.nlargest(5).index.tolist()}")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    im = axes[0].imshow(jaccard_mat, cmap='RdYlGn', vmin=0, vmax=1)
    axes[0].set_xticks(range(0, n, 3))
    axes[0].set_xticklabels(colors[::3], rotation=90, fontsize=6)
    axes[0].set_yticks(range(0, n, 3))
    axes[0].set_yticklabels(colors[::3], fontsize=6)
    axes[0].set_title('Color × Color Jaccard Similarity (co-occurrence across counties)')
    plt.colorbar(im, ax=axes[0])
    axes[1].hist(vocab_size, bins=20, color='steelblue', edgecolor='black')
    axes[1].set_xlabel('Vocabulary size (unique colors)')
    axes[1].set_ylabel('Number of counties')
    axes[1].set_title('County Vocabulary Size Distribution')
    plt.tight_layout()
    p5 = FIGURES_DIR / 'eda_color_presence_vocabulary.png'
    plt.savefig(p5, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p5.relative_to(PROJECT_ROOT)).replace('\\', '/'))


### 3.1 Color Presence Matrix

Vocabulary size: min=9, max=15, mean=12.3

### 3.2 County Vocabulary Size

Top 5 counties by vocabulary: [6011, 6027, 6037, 6049, 6051]


## 4) Conditional Distribution Diagnostics

### 4.1 Expected vs Observed & 4.2 Divergence vs Exposure

In [47]:
# 4.1 Expected vs Observed & 4.2 Divergence vs Exposure
jsd_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'jsd_conditional_divergence.csv'
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
if jsd_path.exists() and exp_by_county.exists():
    jsd_df = pd.read_csv(jsd_path)
    cty_div = jsd_df.groupby('fips')['divergence'].mean().reset_index()
    exp_df = pd.read_csv(exp_by_county)
    exp_df['fips'] = exp_df['county_fips'].astype(str).str.zfill(5)
    cty_div['fips'] = cty_div['fips'].astype(str).str.zfill(5)
    merged = cty_div.merge(exp_df, on='fips', how='inner')
    section('4.1 Expected vs Observed Divergence', 3)
    log(f"Mean county divergence: {merged['divergence'].mean():.4f}")
    log(f"Counties with largest residual: {merged.nlargest(3, 'divergence')['fips'].tolist()}")
    section('4.2 Divergence vs Exposure', 3)
    corr_exp = merged['divergence'].corr(merged['total_exposure'])
    log(f"Correlation divergence vs total_exposure: {corr_exp:.3f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(merged['divergence'], bins=20, color='purple', alpha=0.7)
    axes[0].set_xlabel('Mean divergence')
    axes[0].set_title('Distribution of County Divergence')
    axes[1].scatter(merged['total_exposure'], merged['divergence'], alpha=0.7)
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Total exposure')
    axes[1].set_ylabel('Mean divergence')
    axes[1].set_title(f'Divergence vs Exposure (r={corr_exp:.3f})')
    z = np.polyfit(np.log10(merged['total_exposure']+1), merged['divergence'], 1)
    x_line = np.logspace(2, 6, 100)
    axes[1].plot(x_line, np.poly1d(z)(np.log10(x_line+1)), 'r--', label='trend')
    plt.tight_layout()
    p6 = FIGURES_DIR / 'eda_divergence_exposure.png'
    plt.savefig(p6, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p6.relative_to(PROJECT_ROOT)).replace('\\', '/'))


### 4.1 Expected vs Observed Divergence

Mean county divergence: 0.5869
Counties with largest residual: ['06043', '06003', '06011']

### 4.2 Divergence vs Exposure

Correlation divergence vs total_exposure: -0.502


## 5) High-Exposure Counties Only (Robustness EDA)

Filter: total_structures > 100k, <30% sparse cells. Repeat divergence and vocabulary.

In [40]:
# 5 High-Exposure Robustness
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
jsd_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'jsd_conditional_divergence.csv'
bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
if exp_by_county.exists() and jsd_path.exists() and bay_agg.exists():
    exp_df = pd.read_csv(exp_by_county)
    exp_df['fips'] = exp_df['county_fips'].astype(str).str.zfill(5)
    high_exp = exp_df[exp_df['total_exposure'] > 100000]
    if 'pct_sparse' in exp_df.columns:
        high_exp = high_exp[high_exp['pct_sparse'] < 30]
    jsd_df = pd.read_csv(jsd_path)
    jsd_df['fips'] = jsd_df['fips'].astype(str).str.zfill(5)
    cty_div = jsd_df.groupby('fips')['divergence'].mean()
    df_bay = pd.read_csv(bay_agg)
    df_bay['fips'] = df_bay['fips'].astype(str).str.zfill(5)
    vocab = df_bay.groupby('fips')['clr'].nunique()
    section('5 High-Exposure Counties (Robustness)', 3)
    log(f"Counties with >100k structures: {len(high_exp)}")
    for _, r in high_exp.head(10).iterrows():
        f = r['fips']
        div = cty_div.get(f, np.nan)
        v = vocab.get(f, np.nan)
        log(f"  FIPS {f}: divergence={div:.4f}, vocab_size={v}")
    log("**Interpretation**: If divergence and vocabulary variation remain among high-exposure counties, reporting differences persist beyond sparsity.")


### 5 High-Exposure Counties (Robustness)

Counties with >100k structures: 6
  FIPS 06037: divergence=0.5813, vocab_size=15
  FIPS 06067: divergence=0.4366, vocab_size=11
  FIPS 06059: divergence=0.4198, vocab_size=11
  FIPS 06065: divergence=0.4038, vocab_size=11
  FIPS 06071: divergence=0.4084, vocab_size=11
  FIPS 06073: divergence=0.5715, vocab_size=15
**Interpretation**: If divergence and vocabulary variation remain among high-exposure counties, reporting differences persist beyond sparsity.


## 6) Structured Case Studies (3 Counties)

LA (6037), SD (6073), Alpine (6003) - exposure, landcover, top colors, divergence.

In [41]:
# 6 Case Studies: LA (6037), SD (6073), Alpine (6003)
case_fips = ['6037', '6073', '6003']
bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
jsd_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'jsd_conditional_divergence.csv'
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
if bay_agg.exists() and jsd_path.exists() and exp_by_county.exists():
    df_bay = pd.read_csv(bay_agg)
    df_bay['fips'] = df_bay['fips'].astype(str).str.zfill(5)
    jsd_df = pd.read_csv(jsd_path)
    jsd_df['fips'] = jsd_df['fips'].astype(str).str.zfill(5)
    exp_df = pd.read_csv(exp_by_county)
    exp_df['county_fips'] = exp_df['county_fips'].astype(str).str.zfill(5)
    section('6 Case Studies', 3)
    case_rows = []
    for f in case_fips:
        sub = df_bay[df_bay['fips'] == f]
        exp = exp_df[exp_df['county_fips'] == f]['total_exposure'].values
        exp_val = exp[0] if len(exp) > 0 else 0
        lc_comp = sub.groupby('lc_type')['count'].sum()
        top5 = sub.groupby('clr')['count'].sum().nlargest(5)
        top5_urban = sub[sub['lc_type'] == 'urban'].groupby('clr')['count'].sum().nlargest(5) if 'urban' in sub['lc_type'].values else pd.Series()
        div = jsd_df[jsd_df['fips'] == f]['divergence'].mean()
        case_rows.append({'fips': f, 'exposure': exp_val, 'top5_colors': ', '.join(top5.index[:5]), 'divergence': div})
        log(f"  {f}: exposure={exp_val:,}, top5={list(top5.index[:5])}, divergence={div:.4f}")
    case_df = pd.DataFrame(case_rows)
    log(f"\n**Case comparison table:**\n{case_df.to_string()}")


### 6 Case Studies

  6037: exposure=0, top5=[], divergence=nan
  6073: exposure=0, top5=[], divergence=nan
  6003: exposure=0, top5=[], divergence=nan

**Case comparison table:**
   fips  exposure top5_colors  divergence
0  6037         0                     NaN
1  6073         0                     NaN
2  6003         0                     NaN


## 8) Statistical Tests

Chi-square (clr × county), correlation (divergence vs exposure), ANOVA (vocabulary across exposure bins).

In [42]:
# 8 Statistical Tests
from scipy import stats

section('8 Statistical Tests', 3)
test_results = []

if df_full is not None and 'clr' in df_full.columns and 'fips' in df_full.columns:
    ct = pd.crosstab(df_full['clr'], df_full['fips'])
    chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
    test_results.append(f"Chi-square clr × county: chi2={chi2:.2f}, p={p_chi:.2e}, dof={dof}")
    log(f"Chi-square (clr × county): chi2={chi2:.2f}, p={p_chi:.2e}")

jsd_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'jsd_conditional_divergence.csv'
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
if jsd_path.exists() and exp_by_county.exists():
    jsd_df = pd.read_csv(jsd_path)
    jsd_df['fips'] = jsd_df['fips'].astype(str).str.zfill(5)
    cty_div = jsd_df.groupby('fips')['divergence'].mean().reset_index()
    exp_df = pd.read_csv(exp_by_county)
    exp_df['fips'] = exp_df['county_fips'].astype(str).str.zfill(5)
    merged = cty_div.merge(exp_df, on='fips', how='inner')
    r, p_corr = stats.pearsonr(merged['divergence'], np.log10(merged['total_exposure']+1))
    test_results.append(f"Correlation divergence vs log(exposure): r={r:.3f}, p={p_corr:.2e}")
    log(f"Correlation divergence vs log(exposure): r={r:.3f}, p={p_corr:.2e}")

bay_agg = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'
if bay_agg.exists() and exp_by_county.exists():
    df_bay = pd.read_csv(bay_agg)
    df_bay['fips'] = df_bay['fips'].astype(str).str.zfill(5)
    vocab = df_bay.groupby('fips')['clr'].nunique().reset_index(name='vocab_size')
    exp_df = pd.read_csv(exp_by_county)
    exp_df['fips'] = exp_df['county_fips'].astype(str).str.zfill(5)
    vocab = vocab.merge(exp_df[['fips', 'total_exposure']], on='fips', how='inner')
    try:
        vocab['exp_bin'] = pd.qcut(vocab['total_exposure'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
        groups = [vocab[vocab['exp_bin']==q]['vocab_size'].values for q in ['Q1','Q2','Q3','Q4']]
        groups = [g for g in groups if len(g) > 0]
        f_val, p_anova = stats.f_oneway(*groups) if len(groups) > 1 else (np.nan, np.nan)
    except Exception:
        f_val, p_anova = np.nan, np.nan
    test_results.append(f"ANOVA vocab across exposure quartiles: F={f_val:.2f}, p={p_anova:.2e}")
    log(f"ANOVA vocab across exposure quartiles: F={f_val:.2f}, p={p_anova:.2e}")


### 8 Statistical Tests

Chi-square (clr × county): chi2=6792924.64, p=0.00e+00
Correlation divergence vs log(exposure): r=-0.794, p=1.01e-13
ANOVA vocab across exposure quartiles: F=3.53, p=2.07e-02


## 9. Generate Plots

In [44]:
import matplotlib.pyplot as plt

# figures_generated initialized in setup; EDA cells may have appended
# Plot 1: Landcover distribution from main dataset
if df_main is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    lc_counts = df_main['lc_type'].value_counts()
    ax.barh(range(len(lc_counts)), lc_counts.values, color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(lc_counts)))
    ax.set_yticklabels(lc_counts.index, fontsize=9)
    ax.set_xlabel('Count')
    ax.set_title('Landcover Distribution (Main Dataset Sample)')
    plt.tight_layout()
    p1 = FIGURES_DIR / 'landcover_distribution.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p1.relative_to(PROJECT_ROOT)))

# Plot 2: Exposure distribution from results
exp_h3_path = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_per_h3.csv'
if exp_h3_path.exists():
    df_exp = pd.read_csv(exp_h3_path)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df_exp['total_exposure'], bins=50, color='forestgreen', alpha=0.7, edgecolor='black')
    ax.axvline(df_exp['total_exposure'].median(), color='red', linestyle='--', label=f"Median: {df_exp['total_exposure'].median():.0f}")
    ax.set_xlabel('Structures per H3 cell')
    ax.set_ylabel('Count')
    ax.set_title('Exposure Distribution per H3 Cell')
    ax.legend()
    plt.tight_layout()
    p2 = FIGURES_DIR / 'exposure_distribution.png'
    plt.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p2.relative_to(PROJECT_ROOT)))

# Plot 3: KL divergence distribution from conditional probability
cp_summary = PROJECT_ROOT / 'results' / 'tables' / 'conditional_probability' / 'm01_neighbor_pool_county_lc_summary.csv'
if cp_summary.exists():
    df_cp = pd.read_csv(cp_summary)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(df_cp['kl_div'], bins=30, color='purple', alpha=0.7)
    axes[0].set_xlabel('KL Divergence')
    axes[0].set_title('KL Divergence Distribution')
    axes[1].hist(df_cp['l1_distance'], bins=30, color='teal', alpha=0.7)
    axes[1].set_xlabel('L1 Distance')
    axes[1].set_title('L1 Distance Distribution')
    plt.tight_layout()
    p3 = FIGURES_DIR / 'conditional_probability_metrics.png'
    plt.savefig(p3, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p3.relative_to(PROJECT_ROOT)))

# Plot 4: EDA diagnostic - divergence vs exposure
exp_by_county = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
jsd_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'jsd_conditional_divergence.csv'
if exp_by_county.exists() and jsd_path.exists():
    exp_df = pd.read_csv(exp_by_county)
    jsd_df = pd.read_csv(jsd_path)
    exp_df = exp_df.rename(columns={'county_fips': 'fips'}) if 'county_fips' in exp_df.columns else exp_df
    m = jsd_df.merge(exp_df, on='fips', how='inner')
    if len(m) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.scatter(m['total_exposure'], m['divergence'], alpha=0.6, s=50)
        ax.set_xlabel('County total exposure')
        ax.set_ylabel('JSD divergence')
        ax.set_title('Divergence vs Exposure (sparsity-driven anomaly diagnostic)')
        ax.set_xscale('log')
        plt.tight_layout()
        p4 = FIGURES_DIR / 'divergence_vs_exposure.png'
        plt.savefig(p4, dpi=150, bbox_inches='tight')
        plt.close()
        figures_generated.append(str(p4.relative_to(PROJECT_ROOT)).replace('\\', '/'))

# Plot 5: Exposure vs color diversity
exp_diversity = PROJECT_ROOT / 'results' / 'tables' / '02_exposure_density_sparsity' / 'eda_exposure_diversity.csv'
if exp_diversity.exists():
    df = pd.read_csv(exp_diversity)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(df['total_exposure'], df['n_colors'], alpha=0.3, s=5)
    ax.set_xlabel('Exposure (structures per H3)')
    ax.set_ylabel('Number of color categories')
    ax.set_title('Exposure vs Color Diversity (sparsity artifact)')
    plt.tight_layout()
    p5 = FIGURES_DIR / 'exposure_vs_diversity.png'
    plt.savefig(p5, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p5.relative_to(PROJECT_ROOT)).replace('\\', '/'))

log('\n### Figures Generated')
for p in figures_generated:
    log(f"- `{p}`")



### Figures Generated
- `figures\in_depth_analysis\landcover_distribution.png`
- `figures\in_depth_analysis\exposure_distribution.png`
- `figures\in_depth_analysis\conditional_probability_metrics.png`
- `figures/in_depth_analysis/divergence_vs_exposure.png`
- `figures/in_depth_analysis/exposure_vs_diversity.png`
- `figures/in_depth_analysis/most_common_color_by_county_map.png`
- `figures/in_depth_analysis/most_common_color_by_county_map.png`
- `figures/in_depth_analysis/most_common_color_by_county_map.png`
- `figures/in_depth_analysis/most_common_color_by_county_map.png`
- `figures/in_depth_analysis/most_common_color_by_county_map.png`
- `figures/in_depth_analysis/most_common_color_by_county_map.png`
- `figures/in_depth_analysis/eda_divergence_exposure.png`
- `figures/in_depth_analysis/eda_county_lc_heatmap_structures.png`
- `figures/in_depth_analysis/eda_county_reliability_map.png`
- `figures/in_depth_analysis/eda_exposure_bin_diversity.png`
- `figures/in_depth_analysis/eda_landcover

## 10. Write to results.md

## 11. Most Common Building Color by County (Map)

Visualization of the most common building color for each California county, using mode color from `modes.csv`.

In [45]:
# Map: Most common building color per county (based on notebooks/eda/02_exposure_density_sparsity.ipynb)
import geopandas as gpd
import matplotlib.patches as mpatches

COLOR_HEX = {
    'cocoa': '#8B4513', 'brown': '#8B4513', 'coffee': '#6F4E37', 'tan': '#D2B48C',
    'beige': '#F5F5DC', 'verde': '#228B22', 'green': '#228B22', 'gray': '#808080',
    'grey': '#808080', 'red': '#DC143C', 'orange': '#FF8C00', 'blue': '#4169E1',
    'yellow': '#FFD700', 'white': '#FFFFFF', 'ivory': '#FFFFF0', 'cream': '#FFFDD0',
    'black': '#1a1a1a', 'sienna': '#A0522D', 'terracotta': '#E2725B', 'olive': '#808000',
    'navy': '#000080', 'maroon': '#800000', 'purple': '#800080', 'gold': '#FFD700',
    'auburn': '#A52A2A', 'sage': '#9DC183', 'emerald': '#50C878', 'lavender': '#E6E6FA',
}
modes_path = PROJECT_ROOT / 'notebooks' / 'eda' / 'data' / 'modes.csv'
if modes_path.exists():
    df_modes = pd.read_csv(modes_path)
    df_modes['county_fips'] = df_modes['fips'].astype(str).str.zfill(5)
    df_modes['color_hex'] = df_modes['clr'].map(lambda x: COLOR_HEX.get(str(x).lower(), '#CCCCCC') if pd.notna(x) else '#CCCCCC')
    counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_20m.zip")
    counties_ca = counties[counties["STATEFP"] == "06"].copy()
    counties_ca["county_fips"] = counties_ca["GEOID"]
    gdf_color = counties_ca.merge(df_modes[['county_fips', 'clr', 'color_hex']], on="county_fips", how="left")
    fig, ax = plt.subplots(figsize=(10, 10))
    gdf_color.plot(ax=ax, color=gdf_color['color_hex'], edgecolor='black', linewidth=0.3)
    top6 = df_modes['clr'].value_counts().head(6).index.tolist()
    rows = [df_modes[df_modes['clr'] == c].iloc[0] for c in top6]
    patches = [mpatches.Patch(color=r['color_hex'], label=r['clr']) for r in rows]
    leg = ax.legend(handles=patches, loc='lower left', bbox_to_anchor=(0.02, 0.02), ncol=2, fontsize=14,
                    frameon=True, handlelength=1.5, handleheight=2, labelspacing=0.9)
    ax.set_title('Most Common Building Color by County (California)')
    ax.axis('off')
    plt.tight_layout()
    p_map = FIGURES_DIR / 'most_common_color_by_county_map.png'
    plt.savefig(p_map, dpi=150, bbox_inches='tight')
    plt.close()
    figures_generated.append(str(p_map.relative_to(PROJECT_ROOT)).replace('\\', '/'))
    log(f"\n**Map saved**: {p_map.relative_to(PROJECT_ROOT)}")
else:
    log("modes.csv not found; skipping color map.")


**Map saved**: figures\in_depth_analysis\most_common_color_by_county_map.png


In [ ]:
section('In-Depth Analysis Context (Generated)', 2)
log(f"**Generated**: {datetime.now().isoformat()}")
log("")
log("### Dataset Inventory")
log("")
log("| Location | Tables |")
log("|----------|--------|")
log("| dataset/ | Capstone2025_nsi_lvl9_with_landcover_and_color.csv |")
log("| results/tables/02_exposure_density_sparsity/ | eda_exposure_per_h3, eda_exposure_by_county, eda_exposure_by_landcover, eda_exposure_diversity, eda_sparsity_regimes |")
log("| results/tables/03_color/ | color_similarity_matrix, landcover_color_combinations |")
log("| results/tables/bayesian_shrinkage/ | bayesian_shrinkage_baseline_distributions, bayesian_shrinkage_stabilized_distributions, bayesian_shrinkage_aggregated_counts |")
log("| results/tables/conditional_probability/ | m01_neighbor_pool_county_lc_summary, m01_neighbor_pool_county_lc_color_detail |")
log("| results/tables/morans_i/ | relative_frequencies_lc_type_bldgtype, morans_i_homogeneity |")
log("| website/backend/data/ | All above + ca_county_neighbors, c2st_results, jsd_*, color_*, morans_i_homogeneity |")
log("")
log("### Figures")
for p in figures_generated:
    log(f"- ![{p}]({p})")

context_block = '\n'.join(analysis_output)

# Read existing results.md
if OUTPUT_MD.exists():
    existing = OUTPUT_MD.read_text(encoding='utf-8')
    # Remove previous in-depth block if exists
    if '## In-Depth Analysis Context (Generated)' in existing:
        existing = existing.split('## In-Depth Analysis Context (Generated)')[0].rstrip()
    new_content = existing + '\n\n---\n\n' + context_block
else:
    new_content = '# EDA Results Summary\n\n' + context_block

OUTPUT_MD.write_text(new_content, encoding='utf-8')
print(f"\nWritten to {OUTPUT_MD}")